# 04 · Skill-player availability
Team stats already include the production of the players who have been playing. What they miss is **who is out today**. `src/players.py` measures the share of a team's usual RB/WR/TE touches that is unavailable for each game.

- **Player value = role size:** his share of the team's skill-player targets + carries in earlier games. EPA per game was tried first and gave no gain; it is noisy and mixes in the QB.
- **Participation:** how regularly he has been playing for this team (fades while he's out).
- **Missing:** on the roster but not playing. For upcoming games this uses the injury report (Out/Doubtful) and roster status (IR).

In [1]:
import sys; sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, matplotlib.pyplot as plt
from src.config import PROCESSED_DIR
from src.features import load_player_inputs, FEATURE_COLUMNS
from src.players import skill_availability
from src.tune_features import load_params

p = load_params()
tg = pd.read_parquet(PROCESSED_DIR / 'team_games.parquet').sort_values(['gameday', 'game_id']).reset_index(drop=True)
log, rosters, outs = load_player_inputs()
feat, details = skill_availability(tg, log, rosters, outs, p.player_decay, p.player_season_decay,
                                   p.player_prior_games, p.part_decay, p.part_season_decay, return_details=True)
details = details.merge(tg[['game_id', 'team', 'season', 'week']], on=['game_id', 'team'])

## Biggest absences of 2025
`value` = share of team touches he usually gets, `missing_value` = value × participation.

In [2]:
details[details.season == 2025].sort_values('missing_value', ascending=False)[
    ['week', 'team', 'name', 'position', 'value', 'participation', 'missing_value']].head(15).round(3)

,week,team,name,position,value,participation,missing_value
7627,18,PHI,Saquon Barkley,RB,0.226,0.861,0.194
1026,18,MIA,De'Von Achane,RB,0.206,0.863,0.178
9843,18,NYJ,Breece Hall,RB,0.193,0.860,0.166
21708,18,GB,Josh Jacobs,RB,0.172,0.803,0.138
10823,17,CLE,Quinshon Judkins,RB,0.176,0.771,0.136
21684,12,GB,Josh Jacobs,RB,0.183,0.740,0.135
33454,18,MIN,Aaron Jones,RB,0.179,0.742,0.132
18364,18,DAL,Javonte Williams,RB,0.155,0.815,0.126
10829,18,CLE,Quinshon Judkins,RB,0.176,0.694,0.122
12019,13,NO,Alvin Kamara,RB,0.147,0.742,0.109


## Does a missing-player gap move win rates?
Home win rate by the home-minus-away missing share (positive = home team missing more).

In [3]:
g = pd.read_parquet(PROCESSED_DIR / 'games_features.parquet')
g = g[g.home_win.notna() & (g.season >= 2007)]
b = pd.qcut(g.diff_skill_missing, 5, duplicates='drop')
g.groupby(b, observed=True).home_win.agg(['mean', 'size']).round(3)

,mean,size
diff_skill_missing,,
"(-0.4, -0.028]",0.597,1037
"(-0.028, -0.00506]",0.583,1036
"(-0.00506, 0.00489]",0.551,1036
"(0.00489, 0.029]",0.548,1036
"(0.029, 0.34]",0.515,1036


## With vs. without player features (walk-forward)

In [4]:
from src.evaluate import load_played_games, walk_forward, compare
from src.models import logistic
games = load_played_games()
without = [c for c in FEATURE_COLUMNS if 'skill' not in c]
for label, seasons in [('tuning 2012-18', range(2012, 2019)), ('holdout 2019-25', range(2019, 2026))]:
    P = {'with_players': walk_forward(games, lambda: logistic(0.003), FEATURE_COLUMNS, seasons),
         'without': walk_forward(games, lambda: logistic(0.003), without, seasons)}
    print(label); display(compare(games, P).round(4))

tuning 2012-18


,accuracy,log_loss,brier,games
vegas,0.6658,0.6101,0.2113,1861
with_players,0.6615,0.6165,0.2140,1861
without,0.6577,0.6172,0.2144,1861
elo,0.6550,0.6248,0.2181,1861


holdout 2019-25


,accuracy,log_loss,brier,games
vegas,0.6638,0.6083,0.2105,1954
with_players,0.6561,0.6282,0.2191,1954
without,0.6474,0.6295,0.2199,1954
elo,0.6382,0.6367,0.2227,1954
